# Unity Catalog Governance Setup
**Mục tiêu:** Phân quyền truy cập dựa trên Role-Matrix.
- **Roles:** Data Engineer, Analytics Engineer, Data Analyst, Data Scientist, BI User.
- **Layers:** Bronze, Silver, Gold, Business.

**Quy tắc Mapping (Dựa trên hình ảnh):**
- ✅ (Green Check): `ALL PRIVILEGES` (Toàn quyền: Đọc, Ghi, Xóa, Tạo).
- 👀 (Eyes): `SELECT` (Chỉ đọc).
- ⚠️ (Warning - DE on Gold): `SELECT` + `MODIFY` (Được sửa dữ liệu/debug nhưng hạn chế xóa/thay đổi cấu trúc, hoặc cấp Full quyền nhưng đánh dấu cần cẩn trọng. Ở đây ta sẽ cấp `ALL PRIVILEGES` để pipeline chạy được).
- ❌ (Cross): Không cấp quyền.

In [0]:
# Danh sách các nhóm cần tạo
groups = [
    "data_engineers",
    "analytics_engineers",
    "data_analysts",
    "data_scientists",
    "bi_users"
]

# Tạo Group trong Unity Catalog (Nếu chưa tồn tại)
for group in groups:
    try:
        spark.sql(f"CREATE GROUP IF NOT EXISTS `{group}`")
        print(f"✅ Group created/verified: {group}")
    except Exception as e:
        print(f"⚠️ Warning (Có thể bỏ qua nếu không phải Admin): {group} - {str(e)}")

In [0]:
catalog_name = "brazilian_ecommerce"

# 1. Cấp quyền USE CATALOG cho tất cả các nhóm (Vé vào cổng)
# Nếu không có quyền này, họ thậm chí không nhìn thấy Catalog
all_groups = ["data_engineers", "analytics_engineers", "data_analysts", "data_scientists", "bi_users"]

for group in all_groups:
    spark.sql(f"GRANT USE CATALOG ON CATALOG `{catalog_name}` TO `{group}`")
    # Cấp quyền xem thông tin Catalog (Browse)
    spark.sql(f"GRANT USE SCHEMA ON CATALOG `{catalog_name}` TO `{group}`") 

print("✅ Đã cấp quyền 'USE CATALOG' cho toàn bộ các nhóm.")

In [0]:
# Kiểm tra quyền trên Schema Business (Nơi giao thoa nhiều quyền nhất)
print("🔍 --- QUYỀN TRUY CẬP TRÊN SCHEMA BUSINESS ---")
display(spark.sql(f"SHOW GRANTS ON SCHEMA `{catalog_name}`.`business`"))

print("\n🔍 --- QUYỀN TRUY CẬP TRÊN SCHEMA GOLD ---")
display(spark.sql(f"SHOW GRANTS ON SCHEMA `{catalog_name}`.`gold`"))